# Step 8 — train the selected Transformer on 15-minute Nifty 50 bars

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. **Runtime → Run all.**
3. When cell 3 shows an upload button, choose `regime_15min_bundle.zip`.
4. If Google Drive asks for permission, allow it. Files are kept in Drive, so a disconnect does not lose finished seeds; just Run all again and it resumes.
5. The last cell downloads `tuned_15min_outputs.zip`. Put it in the `redo` folder on your laptop.

Expected time on a T4: roughly 20–40 minutes for 5 seeds.

In [ ]:
# 1. Check the GPU is attached (should print a Tesla T4 or similar)
!nvidia-smi -L

In [ ]:
# 2. Keep files on Google Drive so a disconnect does not lose finished seeds
USE_DRIVE = True
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/nifty_regime_redo'
else:
    WORK = '/content/nifty_regime_redo'
os.makedirs(WORK, exist_ok=True)
print('working directory:', WORK)

In [ ]:
# 3. Upload the bundle once (skipped if it is already unpacked)
import zipfile
from google.colab import files
if not os.path.exists(os.path.join(WORK, 'scripts', '08_train_15min.py')):
    uploaded = files.upload()   # choose regime_15min_bundle.zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(WORK)
os.chdir(WORK)
print(sorted(os.listdir(WORK)))

In [ ]:
# 4. Dependencies (torch, pandas, numpy, scikit-learn are preinstalled on Colab)
!pip -q install pyyaml pyarrow

In [ ]:
# 5. Train 5 seeds on GPU (resumes finished seeds)
!python scripts/08_train_15min.py --device cuda

In [ ]:
# 6. Download the trained weights and validation report
out = 'tuned_15min_outputs.zip'
with zipfile.ZipFile(out, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in ['models/tuned_15min']:
        for f in sorted(os.listdir(folder)):
            z.write(os.path.join(folder, f))
    for f in ['reports/08_15min_validation.json', 'reports/08_15min_validation.md']:
        z.write(f)
print(open('reports/08_15min_validation.md').read())
files.download(out)